## If working in Google Colab, Run this first
Run the cell below, at the start of every session. 
It downloads the course files to your Google Drive and keeps them up to date.
When you see \"Setup complete\", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.** 
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
os.chdir('/content')                      # start from a valid directory
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/AIforMovementScience'
if os.path.isdir(REPO + '/.git'):
    # already downloaded before: just update it
    !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
else:
    # first time, or a broken/partial copy: download a fresh copy
    !rm -rf "$REPO"
    !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"

WORKDIR = REPO + '/lectures'          # this notebook's folder in the course files
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)           # so local .py files (e.g. FunctionExamples.py) import correctly
print('Setup complete. You can now run the rest of the notebook.')

## Classification

In this notebook, we will introduce the concept of classification and learn how to apply our knowledge of linear regression to address this new machine learning challenge.

In linear regression, we found a relationship between independent variables (inputs) and dependent variables (outputs). The input variables can be continuous (e.g., some measurement from a sensor) or discrete, and the output is a continuous variable. 

Similarly, in classification, we are interested in finding a relationship between independent variables (features) and a dependent variable (classes). However, in this case, the classes must be a discrete variable with just two or a few potential values. 

| feature 1 | feature 2 | Class|
|--|--|--|
|0.25 | 0.13| 0|
|0.45 | 0.23| 0|
|0.12 | 0.98| 1|
|0.35 | 0.16| 0|
|0.05 | 0.74| 1|
|0.36 | 0.08| 0|


The objective of the classification problem is to identify the relationship between the features and the classes, such that when new features are available, it is possible to predict the class.

| feature 1 | feature 2 | Class|
|--|--|--|
|0.07 | 0.78| ?|

Note that the class can be a category (i.e., belongs to a group such as SEX (M, class=0, F, class=1), or disease (Healthy, class=0, patient, class=1)) or can be derived by grouping continuous variables (i.e, groups created from age, such as young (age <18, class=0), adult (age >=18 and <60, class=1), older adult (age >=60, class=2))

When you have just two or three features, it is possible to visualize your data.


In [ ]:
from sklearn.datasets import make_blobs
import numpy as np
import matplotlib.pyplot as plt


# create 50 separable points
X, y = make_blobs(n_samples=50, centers=2,
                  random_state=0, cluster_std=0.60)

# plot the data
fig, ax = plt.subplots(figsize=(5, 5))
point_style = dict(cmap='Paired', s=50)
ax.scatter(X[:, 0], X[:, 1], c=y, **point_style)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

Here we have two-dimensional data: that is, we have two features for each point, represented by the $(x,y)$ positions of the points on the plane. In addition, we have one of two class labels for each point (class 0 -> Blue, class 1 -> Brown). 

From these features and labels, we would like to create a model that will help us decide whether a new point should be labeled "blue" or "brown."

In [ ]:
#now, if we have a new datapoint, how do we determine the class?
# plot the data
fig, ax = plt.subplots(figsize=(5, 5))
point_style = dict(cmap='Paired', s=50)
ax.scatter(X[:, 0], X[:, 1], c=y, **point_style)
ax.scatter(1,2.8,c='red')
ax.text(1.05,2.85,'?', fontsize=20)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

There are a number of possible models for such a classification task. We will use an extremely simple one. We will make the assumption that the two groups can be separated by drawing a straight line through the plane between them, such that points on each side of the line fall in the same group. 

Here, the model is a quantitative version of the statement "a straight line separates the classes". The objective is then to use the data to learn the intercept and slope of this line and use it to make predictions about new data points. At the end, the optimal values for these model parameters are learned from the data using gradient descent or a related algorithm. 

---
Going back to our linear regression notation, we have that 
$$
\hat{y} = \omega X
$$

where $X$ is the feature matrix and $\omega$ the model parameters (intercept and slope). The model parameters are obtained by comparing  the model predictions ($\hat{y}$) and the classes ($y$).


### The Problem
This is an ill-defined problem. By definition, the classes are discrete (0 or 1 in this case), but the model output ($\hat{y}$) can take any value, either positive or negative. Thus, there is no easy way to compare the model output and the measured classes. 

One potential solution to this problem is to transform the model output so that it becomes 0 or 1 (as the classes). There are many ways to do this (such as using a threshold); however, if we want to use gradient descent for parameter estimation, we need to ensure that we can easily estimate the gradient of the resulting function. 

### The solution 
One reasonable transformation is to use the sigmoid function, which maps any value $x$ into a new value in the range 0 and 1. 
$$
\sigma(x) = \frac{1}{1+e^{-x}}
$$

In [ ]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

x_sig = np.linspace(-10,10,100)
plt.plot(x_sig, sigmoid(x_sig))
plt.ylabel('Sigmoid(x)')
plt.xlabel('x')

Thus, our new model is 
$$
\hat{y} = \sigma(\omega X)
$$

or

$$
\hat{y} = \frac{1}{1+e^{-\omega X}}
$$

Interestingly, by forcing the output to be between 0 and 1, we can interpret the model's output as the probability of belonging to class 1. So, the model does not estimate the class directly; rather, it estimates how likely the feature set is to belong to class 1.  

### Cost Function 
When dealing with linear regression, we used the mean squared error (MSE) as the cost function. This cost function allowed us to find the optimal parameter set that reduces the error between measurements and predictions. We chose that cost function because it was **CONVEX**, allowing us to find the optimal parameters with gradient descent. 

However, when using the MSE in the classification problem, we obtain
$$
MSE (\omega) = \frac{1}{2N}\sum(\hat{y} - y)^2 \\
$$

$$
MSE (\omega) = \frac{1}{2N}\sum( \frac{1}{1+e^{-\omega X}} - y)^{2}
$$

This function is no longer convex. It can have flat regions and local minima, so gradient descent can stop at a poor set of parameters.

---

Remember that the goal of the cost function is to be near zero when the model makes a good prediction and to be very large when the model makes a bad prediction, so we can use it along with gradient descent to improve the model parameters. 

An alternative **CONVEX** cost function is a **logistic loss**. This function has the following form :
$$
L(\boldsymbol{\omega}) = -\frac{1}{m}\sum_{i=1}^{m}\left[y[i]\log(\hat{y}[i]) + (1-y[i])\log(1-\hat{y}[i])\right]
$$

- $y[i]$ is the true class (0 or 1)
- $\hat{y}[i]$ is the model prediction (a number between 0 and 1)

Let us check this formula: 
- Case 1: $y[i] = 1$ and $\hat{y}[i] \approx 1$ (**Correct Classification**)
    $$
    \begin{align}
        &(1-y[i]) = 0  \rightarrow (1-y[i])\log(1-\hat{y}[i]) = 0 \\
        &\log(\hat{y}[i]) \approx 0  \rightarrow y[i]\log(\hat{y}[i]) \approx 0  \\
    \end{align}
    $$
    
    $$
    L(\boldsymbol{\omega}) \approx 0 
    $$
    
- Case 2: $y[i] = 0$ and $\hat{y}[i] \approx 0$ (**Correct Classification**)
    $$
    \begin{align}
        &y[i] = 0 \rightarrow y[i]\log(\hat{y}[i]) = 0\\
        &\log(1 - \hat{y}[i]) \approx 0 \rightarrow (1-y[i])\log(1-\hat{y}[i]) \approx 0 \\
    \end{align}
    $$
    
    $$
    L(\boldsymbol{\omega}) \approx 0 
    $$
    
- Case 3: $y[i] = 1$ and $0 < \hat{y}[i] < 1$ (**Incorrect Classification**)

    $$
    \begin{align}
        &y[i] = 1 \rightarrow (1-y[i])\log(1-\hat{y}[i]) = 0\\
        &\log(\hat{y}[i]) < 0 \ \text{becomes closer to 0 as it gets closer to 1}  \\
        &\rightarrow y[i]\log(\hat{y}[i]) < 0 \\
    \end{align}
    $$
    
    $$
    L(\boldsymbol{\omega}) > 0  
    $$
    
- Case 4: $y[i] = 0$ and $0<\hat{y}[i] < 1$ (**Incorrect Classification**)

    $$
    \begin{align}
        &y[i] = 0 \rightarrow y[i]\log(\hat{y}[i]) = 0\\
        &\log(1 - \hat{y}[i]) < 0 \text{becomes closer to 0 as it gets closer to 0} \\
        &\rightarrow (1-y[i])\log(1-\hat{y}[i]) < 0 \\
    \end{align}
    $$
    
    $$
    L(\boldsymbol{\omega}) > 0  
    $$
    
So, the **logistic loss** is non-zero when the prediction model generates a probability that does not agree with the training data.


The goal now is to use this cost function as part of a gradient descent iterative algorithm to find the slope and intercept of our model. An algorithm to estimate these parameters is available as part of the `sklearn` library.

In [ ]:
from sklearn.linear_model import LogisticRegression

#fit the model
log_reg = LogisticRegression()
log_reg.fit(X,np.ravel(y))

# plot the data
fig, ax = plt.subplots(figsize=(5, 5))
point_style = dict(cmap='Paired', s=50)
ax.scatter(X[:, 0], X[:, 1], c=y, **point_style)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

#to plot the boundary 
# 2. Extract coefficients
# log_reg.coef_ is a 2D array as there are two features: [[beta1, beta2]]
# log_reg.intercept_ is a 1D array: [beta0]
beta0 = log_reg.intercept_[0]
beta1, beta2 = log_reg.coef_[0]


x_ = np.linspace(-1,3,100)
y_ = -(beta0/beta2 + (beta1/beta2)*x_)
plt.plot(x_,y_)


The line in the plot is the **decision boundary**. On this line the linear model output is zero, so the probability is exactly 0.5:

$$
\beta_0 + \beta_1 x_1 + \beta_2 x_2 = 0 \quad\Rightarrow\quad x_2 = -\frac{\beta_0 + \beta_1 x_1}{\beta_2}
$$

Points on one side of the line have a probability above 0.5 (class 1). Points on the other side have a probability below 0.5 (class 0).

Now, it is possible to use this model to make predictions to for new data point 
- feature 1 = 1.05
- feature 2 = 2.85

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
point_style = dict(cmap='Paired', s=50)
ax.scatter(X[:, 0], X[:, 1], c=y, **point_style)

#to plot boundary
plt.plot(x_,y_)
ax.scatter(1.05,2.85,c='red')
ax.text(1.1,2.9,'?', fontsize=20)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

In [ ]:
# We can predict the output of the linear model
output_LinearModel = (beta0 + 1.05*beta1 + beta2*2.85)
print(f'The output of the linear model is = {output_LinearModel}')

# With this output, we can compute the probability of belonging to class 1 (brown), using the sigmoid function
Probability_of_class_1 = (1/(1+np.exp(-output_LinearModel)))
print(f'The probability of class 1 is = {Probability_of_class_1*100:.3f}%')

# and the probability of class 0 (blue)
print(f'The probability of class 0 is = {(1-Probability_of_class_1)*100:.3f}%')

#and finally, the class, assuming a threshold of 0.5
predicted_class = 1 if Probability_of_class_1 >0.5 else 0
print(f'The predicted class is = {predicted_class}')


You can obtain the same results using pre-built functions

In [ ]:
# We can predict the probability of belonging to either class
prediction_probability = log_reg.predict_proba(np.array([[1.05,2.85]]))
print(f'Probability of belonging to class 0 = {prediction_probability[0][0]*100:.3f}%, class 1 = {prediction_probability[0][1]*100:.3f}%')

# or we can predict the class directly, assuming a threshold of 0.5
prediction_class = log_reg.predict(np.array([[1.05,2.85]]))
print(f'The predicted class is = {prediction_class[0]}')

## Activity

Compute the probabilities for the following input values :
- [1.5, 4.5]
- [2.5, 1]
- [2.499,2.865]

What differences do you notice? 
Hint -> draw the point, see its location relative to the boundary line

## Reading the coefficients: odds and odds ratios

In research papers, the coefficients of a logistic regression are usually reported as **odds ratios**. This section shows what they mean.

### Odds

The **odds** of class 1 compare the probability of class 1 with the probability of class 0:

$$
\text{odds} = \frac{p}{1-p}
$$

| Probability $p$ | Odds | Read as |
| --- | --- | --- |
| 0.50 | 1 | 1 to 1: both classes are equally likely |
| 0.80 | 4 | 4 to 1: class 1 is four times as likely as class 0 |
| 0.20 | 0.25 | 1 to 4: class 0 is four times as likely as class 1 |

### The linear part of the model is the log of the odds

If we take the sigmoid $p = 1/(1+e^{-z})$ and solve for $z$, we get:

$$
\log\left(\frac{p}{1-p}\right) = z = \beta_0 + \beta_1 x_1 + \beta_2 x_2
$$

So the linear model output $z$ is the **log-odds** of class 1. This is why the model is still "linear": it is linear in the log-odds, not in the probability.

### Odds ratio

If $x_1$ increases by 1 unit (and $x_2$ stays the same), $z$ increases by $\beta_1$. So the odds are **multiplied** by $e^{\beta_1}$. This number is the **odds ratio (OR)** of $x_1$:

$$
\text{OR}_1 = e^{\beta_1}
$$

- $\text{OR} > 1$ ($\beta > 0$): the odds of class 1 increase when the feature increases.
- $\text{OR} < 1$ ($\beta < 0$): the odds of class 1 decrease when the feature increases.
- $\text{OR} = 1$ ($\beta = 0$): the feature has no effect.

**Example.** A model predicts a fall in the next year from gait speed in m/s. The coefficient of speed is $\beta = -2.3$. Then $\text{OR} = e^{-2.3} \approx 0.10$ per 1 m/s. A change of 1 m/s is very large, so we usually report a smaller step. For 0.1 m/s: $\text{OR} = e^{-2.3 \times 0.1} \approx 0.79$. Each 0.1 m/s of extra speed multiplies the odds of a fall by 0.79, a 21% decrease.

The code below checks this rule with the model of the two blobs.

In [ ]:
# Step 1: the odds ratio of each feature is e^beta
print(f'feature 1: beta = {beta1:.3f} -> odds ratio = e^beta = {np.exp(beta1):.2f}')
print(f'feature 2: beta = {beta2:.3f} -> odds ratio = e^beta = {np.exp(beta2):.2f}')

In [ ]:
# Step 2: check it. Move a point 1 unit along feature 1 and compare its odds.
before = np.array([[1.05, 2.85]])
after  = np.array([[2.05, 2.85]])     # feature 1 is 1 unit larger; feature 2 is the same

print("Let's see what happens when we move a point 1 unit along feature 1 and compare its odds.")
print(f'feature 1  before= {before[0,0]:.2f}, after = {after[0,0]:.2f}')
print(f'feature 2  before= {before[0,1]:.2f}, after = {after[0,1]:.2f}')

print('The odds ratio tell us that if we move a point 1 unit along feature 1, while keeping other features constant, the odds of belonging to class 1 are multiplied by e^beta1 = {:.2f}'.format(np.exp(beta1)))



print('\n')
print(f'probability:{before} = {p_before:.3f},\n            {after} = {p_after:.3f}')

#compute probabilities and odds for the two points
p_before = log_reg.predict_proba(before)[0, 1]   # row 0, column 1 = probability of class 1
p_after  = log_reg.predict_proba(after)[0, 1]

odds_before = p_before / (1 - p_before)
odds_after  = p_after / (1 - p_after)
print(f'odds changed from {odds_before:.3f} to {odds_after:.3f}')
print(f'the odds were multiplied by {odds_after / odds_before:.2f}, the same as e^beta1')

### Be careful when you read odds ratios

1. **An odds ratio is not a change in probability.** The odds ratio is the same everywhere, but the change in probability depends on where you start. Near $p = 0.5$ a step changes the probability a lot; near $p = 0$ or $p = 1$ it changes it very little (look at the shape of the sigmoid). In papers and news, an OR of 0.38 is often reported as "62% lower risk". This is wrong: the **odds** are 62% lower, but the **probability** (the risk) changes by a different amount that depends on where you start.
2. **The units matter.** The odds ratio is "per 1 unit" of the feature. An OR per 1 m of step length and an OR per 1 cm of step length are very different numbers. If the features are scaled with `StandardScaler`, the OR is "per 1 standard deviation".
3. **"With the other features held constant."** Each OR assumes that the other features do not change. If two features are strongly correlated (for example, step length and speed), this assumption is not realistic, and their ORs can be misleading.
4. **scikit-learn shrinks the coefficients.** By default, `LogisticRegression` adds a small penalty that makes the coefficients smaller than in a standard statistics package. This is good for prediction. To report odds ratios with confidence intervals and p-values in a paper, use a statistics package such as `statsmodels` (`statsmodels.api.Logit`).

## Example with real data: classify sex from gait

Now we use logistic regression on real gait data. The file `data/gait_sex_walk.csv` comes from the Running Injury Clinic dataset (Ferber et al., 2024, *Scientific Data* 11, 1232). Each row is one person walking on a treadmill. It holds only the four columns we use here: `sex`, `height_cm`, `peak_hip_adduction` (deg) and `stride_length` (m). You will use the full dataset in the final project.

**Question:** can we tell if a person is female or male from two gait measures?

- **Peak hip adduction** (deg): how much the thigh moves toward the midline during stance. Women usually show more hip adduction, partly because of a wider pelvis.
- **Stride length**: the distance covered in one stride. Taller people take longer strides, so we **normalize** it by height: stride length as a **percent of height**. Without this step, stride length also tells the model how tall the person is, and men are taller on average. We want the model to learn from gait, not from body size.

We use the same steps as with the blobs: look at the data, fit the model, draw the decision boundary, read the coefficients.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

walk = pd.read_csv('data/gait_sex_walk.csv')
walk['female'] = (walk['sex'] == 'Female').astype(int)                            # class 1 = female, class 0 = male
walk['stride_pct_height (%)'] = 100 * (walk['stride_length'] * 100) / walk['height_cm']   # stride length, % of height

print('number of people:', len(walk))
print(walk.groupby('sex')[['height_cm', 'peak_hip_adduction', 'stride_length', 'stride_pct_height (%)']].mean().round(2))

Look at the table. Men take longer strides in meters, but they are also taller. As a percent of height, the stride length of men and women is almost the same. Hip adduction is clearly different between the groups.

Now we fit the model with the two inputs.

In [ ]:
inputs = ['peak_hip_adduction', 'stride_pct_height (%)']
train, test = train_test_split(walk, test_size=0.25, random_state=42, stratify=walk['female'])

model_sex = LogisticRegression()
model_sex.fit(train[inputs], train['female'])

b0 = model_sex.intercept_[0]
b1, b2 = model_sex.coef_[0]
print(f'intercept = {b0:.3f} | hip adduction: {b1:.3f} per deg | stride length: {b2:.4f} per % of height')

The coefficient of hip adduction is positive: more hip adduction gives a higher probability of female. The coefficient of the normalized stride length is almost zero: it does not help the model.

As with the blobs, the decision boundary is the line where $\beta_0 + \beta_1 x_1 + \beta_2 x_2 = 0$.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for value, name, color in [(0, 'male', 'C0'), (1, 'female', 'C1')]:
    rows = test[test['female'] == value]
    ax.scatter(rows['peak_hip_adduction'], rows['stride_pct_height (%)'], s=12, alpha=0.6, color=color, label=name)

x1 = np.linspace(-5, 18, 100)
x2 = -(b0 + b1 * x1) / b2
ax.plot(x1, x2, 'k', label='decision boundary')
ax.set_xlim(-5, 18)
ax.set_ylim(50, 105)
ax.set_xlabel('Peak hip adduction (deg)')
ax.set_ylabel('Stride length (% of height)')
ax.set_title('Test set')
ax.legend()
plt.show()

The boundary is almost a vertical line: the model decides mostly from hip adduction, and the stride length has almost no effect.

### Odds ratios

As in the section above, the odds ratio of each input is $e^{\beta}$ per unit. For stride length we also report it per 10% of height.

In [ ]:
print(f'hip adduction: odds ratio per 1 degree       = {np.exp(b1):.2f}')
print(f'stride length: odds ratio per 10% of height = {np.exp(10 * b2):.2f}')

- Each extra degree of hip adduction multiplies the odds of female by about 1.4, at the same stride length.
- The odds ratio of the normalized stride length is close to 1: **an odds ratio of 1 means no effect**.

**Why normalize?** If we use the stride length in meters, the model gets a little better, because the stride length then carries information about height. But that is body size, not gait. Always ask what a feature really measures before you trust its odds ratio.

### How good is this model?

The **accuracy** is the fraction of people that the model classifies correctly (`model.score`). Compare it with a simple model that always predicts the larger group (female).

In [ ]:
baseline = max(test['female'].mean(), 1 - test['female'].mean())
print(f'always predict the larger group: {baseline:.3f}')
print(f"train accuracy                 : {model_sex.score(train[inputs], train['female']):.3f}")
print(f"test accuracy                  : {model_sex.score(test[inputs], test['female']):.3f}")

With hip adduction, the model classifies almost 3 of 4 people correctly, clearly better than the simple guess. The test accuracy is not lower than the train accuracy, so the model does not overfit.

The two groups still overlap (look at the plot), so some people are on the "wrong" side of the line. A straight line cannot separate them completely. In the next lectures you will see models that can draw curved boundaries.

## Multiclass Classification

Logistic regression can be naturally extended to more than two classes. The loss function can be expressed as

$$
L(\boldsymbol \omega) = -\frac{1}{N}\sum_{k}\sum_{i}y_{k}[i]\log(\hat{y}_{k}[i])
$$

Where $K$ is the number of classes ($k = 1, \dots, K$), $N$ is the number of samples, $y_{k}[i]$ is the target, $\hat{y}_{k}[i]$ is the predicted probability, and $\boldsymbol \omega$ are the model parameters. 

Note that when $y_{k}[i]$ is 1 (or 0) and  $\hat{y}_{k}[i]$ is close to 1 (or 0), the loss is close to zero. However, when $y_{k}[i]$ is 1 (or 0) and $\hat{y}_{k}[i]$ is close to 0 (or 1), the loss function is large. 

In the multiclass problem, the probability is computed with the **softmax** function, an extension of the sigmoid for more than two classes
$$
\hat{y}_{k}[i] = \mathrm{softmax}(X\boldsymbol{\omega})_{k} = \frac{\exp(X\boldsymbol{\omega_{k}})}{\sum_{j=1}^{K}\exp(X\boldsymbol{\omega_{j}})}
$$

where each class $j$ has its own set of parameters $\boldsymbol{\omega_{j}}$

In [ ]:
# Example

import numpy as np
from sklearn import datasets
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

#load the Iris dataset, a famous dataset for multiclass classification
iris = datasets.load_iris()
X = iris['data'][:,2:] #only two features
y = (iris['target']).astype(int)

plt.figure(figsize=(10, 4))
plt.plot(X[y==0, 0], X[y==0, 1], "bs", label = 'setosa')
plt.plot(X[y==1, 0], X[y==1, 1], "g^", label='versicolor')
plt.plot(X[y==2, 0], X[y==2, 1], "ro", label = 'virginica')
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.legend()
plt.show();

In [ ]:
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15,random_state=random_seed)

log_reg_multiclass = LogisticRegression()
log_reg_multiclass.fit(X_train,y_train)

#for each element in the test set, check the true class and the predicted probabilities
test_pred_proba = log_reg_multiclass.predict_proba(X_test)


train_pred = log_reg_multiclass.predict(X_train)
test_pred = log_reg_multiclass.predict(X_test)

overall_train_accuracy = (train_pred == y_train).mean()
overall_test_accuracy = (test_pred == y_test).mean()

print(f'Overall train accuracy = {overall_train_accuracy:.1%}')
print(f'Overall test accuracy  = {overall_test_accuracy:.1%}')

print('\n')
print(f'Per class accuracy:')
for class_id, class_name in enumerate(iris.target_names):
    train_mask = y_train == class_id
    test_mask = y_test == class_id

    train_accuracy = (train_pred[train_mask] == class_id).mean()
    test_accuracy = (test_pred[test_mask] == class_id).mean()

    print(f'{class_name}: train accuracy = {train_accuracy:.1%}, '
          f'test accuracy = {test_accuracy:.1%}')



    
    


In [ ]:
for idx, results in enumerate(test_pred_proba):

    
    print(f'Petal length = {X_test[idx,0]}, Petal width = {X_test[idx,1]}')
    print(f'True class = {y_test[idx]}')
    print(f'Predicted Probability Class 0 = {results[0]*100:.1f}%')
    print(f'Predicted Probability Class 1 = {results[1]*100:.1f}%')
    print(f'Predicted Probability Class 2 = {results[2]*100:.1f}%')

    print('-----------')